# SFT Notebook — Supervised Fine-Tuning (Self-Contained)

**Semua logic sudah ada di dalam notebook ini. Tidak butuh file `.py` eksternal.**

### Fitur
- Auto-load CPT LoRA jika tersedia
- Real-time W&B monitoring + validation metrics
- Incremental data support
- Label imbalance detection
- Inference test setelah training
- **Merge LoRA → model final 16-bit** (tersimpan di Google Drive)
- **TEST_MODE** — tes pipeline cepat dengan data terbatas

## Step 1: Configuration

In [ ]:
# ============================================
# CONFIGURATION — EDIT BAGIAN INI
# ============================================

# ── TEST MODE ──────────────────────────────
# Set True untuk tes pipeline cepat (~5 menit)
# Set False untuk full training
TEST_MODE = True
MAX_ROWS = 1000        # Jumlah entry maks saat TEST_MODE
TEST_EPOCHS = 1        # Jumlah epoch saat TEST_MODE
# ───────────────────────────────────────────

MODEL_NAME = "unsloth/Qwen3-8B-bnb-4bit"

# W&B
WANDB_PROJECT = "tim1-dfk"
WANDB_RUN_NAME = "sft-run-"
WANDB_ENTITY = None

# Training
NUM_EPOCHS = TEST_EPOCHS if TEST_MODE else 5
BATCH_SIZE = 2
GRADIENT_ACCUMULATION = 4  # effective batch = 2*4 = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LENGTH = 2048
EVAL_STEPS = 50
SAVE_STEPS = 50
VALIDATION_SPLIT = 0.1

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Paths
DRIVE_PROJECT = "/content/drive/MyDrive/Tim1-DFK"
SFT_DATA_DIR = "Dataset/SFT"
CPT_OUTPUT_DIR = "outputs/cpt"
SFT_OUTPUT_DIR = "outputs/sft"

# Options
USE_CPT_LORA = True
RUN_INFERENCE_TEST = True

# ── OUTPUT MODEL ───────────────────────────
# Merge LoRA ke model penuh dan simpan sebagai model akhir
SAVE_MERGED_16BIT = True   # Simpan merged model 16-bit ke Google Drive
# ───────────────────────────────────────────

print("=" * 60)
if TEST_MODE:
    print("⚡ TEST MODE AKTIF — pipeline test dengan data terbatas")
    print(f"  Max rows: {MAX_ROWS}")
else:
    print("🚀 FULL TRAINING MODE")
print("=" * 60)
print(f"  Model: {MODEL_NAME}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Effective Batch Size: {BATCH_SIZE * GRADIENT_ACCUMULATION}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Validation Split: {VALIDATION_SPLIT * 100}%")
print(f"  Use CPT LoRA: {USE_CPT_LORA}")
print(f"  Inference Test: {RUN_INFERENCE_TEST}")
print(f"  W&B Project: {WANDB_PROJECT}")
print(f"  Merge → 16-bit: {SAVE_MERGED_16BIT}")
print("=" * 60)

## Step 2: Install Dependencies

In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q hf_transfer wandb rich tqdm pandas

## Step 3: Setup Environment

In [ ]:
import os, sys, re, json, random, logging
from pathlib import Path
from datetime import datetime

os.environ['HF_HOME'] = '/content/hf_cache'
os.makedirs('/content/hf_cache', exist_ok=True)
os.environ.setdefault('HF_HUB_ENABLE_HF_TRANSFER', '1')

from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = Path(DRIVE_PROJECT)
if not PROJECT_ROOT.exists():
    print(f'ERROR: {DRIVE_PROJECT} tidak ditemukan di Google Drive!')
    sys.exit(1)

os.chdir(PROJECT_ROOT)
print(f'Working directory: {os.getcwd()}')
print(f'Timestamp: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')

## Step 4: Setup W&B

In [ ]:
import wandb

wandb.login()

api = wandb.Api()
viewer = api.viewer
username = viewer.entity if hasattr(viewer, 'entity') else str(viewer)
print(f'W&B connected as: {username}')

os.environ['WANDB_API_KEY'] = wandb.api.api_key
if WANDB_ENTITY:
    os.environ['WANDB_ENTITY'] = WANDB_ENTITY

timestamp = datetime.now().strftime("%m%d_%H%M%S")
WANDB_RUN_FULL = f"{WANDB_RUN_NAME}{timestamp}"
entity_display = WANDB_ENTITY or username
print(f'Run name: {WANDB_RUN_FULL}')
print(f'Monitor at: https://wandb.ai/{entity_display}/{WANDB_PROJECT}')

## Step 5: Pre-Download Model

Skip otomatis jika sudah ter-cache dari CPT notebook.

In [ ]:
from huggingface_hub import snapshot_download
import shutil

MODEL_LOCAL = "/content/hf_cache/" + MODEL_NAME.split("/")[-1]

existing = list(Path(MODEL_LOCAL).rglob("*.safetensors")) if Path(MODEL_LOCAL).exists() else []
if existing:
    total_gb = sum(f.stat().st_size for f in existing) / 1e9
    print(f"Model sudah ter-cache: {total_gb:.2f} GB ({len(existing)} files)")
    print("(Skip download)")
else:
    disk = shutil.disk_usage("/content")
    print(f"Disk tersedia: {disk.free / 1e9:.1f} GB")
    print(f"Downloading {MODEL_NAME}...")
    snapshot_download(repo_id=MODEL_NAME, local_dir=MODEL_LOCAL, resume_download=True)
    files = list(Path(MODEL_LOCAL).rglob("*.safetensors"))
    total_gb = sum(f.stat().st_size for f in files) / 1e9
    print(f"Model downloaded: {total_gb:.2f} GB")

## Step 6: Preprocess SFT Data

Memuat semua CSV dari `Dataset/SFT/`, mengkonversi ke format Alpaca, dan split train/val.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

# ── Label Map & Reasoning Templates ──

LABEL_MAP = {
    "hoax": "Hoax", "hoaks": "Hoax", "fake": "Hoax",
    "false": "Hoax", "palsu": "Hoax",
    "disinformation": "Disinformasi", "disinformasi": "Disinformasi",
    "misinformation": "Misinformasi", "misinformasi": "Misinformasi",
    "misleading": "Misinformasi", "menyesatkan": "Misinformasi",
    "hate speech": "Ujaran Kebencian", "hate_speech": "Ujaran Kebencian",
    "kebencian": "Ujaran Kebencian", "hateful": "Ujaran Kebencian",
    "toxic": "Ujaran Kebencian", "1": "Ujaran Kebencian",
    "fitnah": "Fitnah", "slander": "Fitnah", "defamation": "Fitnah",
    "factual": "Bukan DFK", "true": "Bukan DFK", "benar": "Bukan DFK",
    "fact": "Bukan DFK", "clean": "Bukan DFK", "0": "Bukan DFK",
    "not hate": "Bukan DFK", "normal": "Bukan DFK",
}

REASONING_TEMPLATES = {
    "Hoax": [
        "Teks ini mengandung klaim yang tidak dapat diverifikasi dan bertujuan menyebarkan informasi palsu kepada publik.",
        "Konten ini termasuk hoaks karena menyebarkan narasi yang bertentangan dengan fakta yang telah diverifikasi oleh lembaga resmi.",
        "Informasi dalam teks ini tidak memiliki sumber yang dapat dipercaya dan dirancang untuk menyesatkan pembaca.",
    ],
    "Disinformasi": [
        "Teks ini merupakan disinformasi karena secara sengaja menyebarkan informasi yang salah dengan tujuan tertentu.",
        "Konten ini termasuk disinformasi yang dirancang untuk mempengaruhi opini publik dengan narasi yang menyimpang dari fakta.",
        "Informasi dalam teks ini telah dimanipulasi secara sengaja untuk menciptakan persepsi yang tidak sesuai dengan kenyataan.",
    ],
    "Misinformasi": [
        "Teks ini mengandung misinformasi yang menyebarkan klaim tidak akurat, meskipun mungkin tidak ada niat sengaja untuk menyesatkan.",
        "Konten ini termasuk misinformasi karena mengandung fakta yang keliru atau tidak lengkap yang dapat menyesatkan pembaca.",
        "Informasi dalam teks ini tidak akurat dan dapat menimbulkan kesalahpahaman di masyarakat.",
    ],
    "Ujaran Kebencian": [
        "Teks ini mengandung ujaran kebencian yang menyerang kelompok tertentu berdasarkan identitas SARA (Suku, Agama, Ras, Antar-golongan).",
        "Konten ini termasuk ujaran kebencian karena menggunakan bahasa yang bersifat merendahkan, menghasut, atau mendiskriminasi.",
        "Teks ini mengandung ekspresi yang provokatif dan berpotensi menimbulkan permusuhan terhadap kelompok tertentu.",
    ],
    "Fitnah": [
        "Teks ini mengandung fitnah karena menyebarkan tuduhan palsu yang dapat merusak reputasi seseorang atau kelompok.",
        "Konten ini termasuk fitnah karena mengandung klaim tidak berdasar yang bertujuan mencemarkan nama baik pihak tertentu.",
        "Teks ini merupakan fitnah yang berpotensi melanggar hukum karena menyebarkan informasi palsu yang merugikan pihak lain.",
    ],
    "Bukan DFK": [
        "Teks ini bukan termasuk konten DFK. Konten ini berisi informasi yang dapat diverifikasi dan tidak mengandung unsur disinformasi, fitnah, atau kebencian.",
        "Berdasarkan analisis, teks ini tidak mengandung elemen DFK. Konten ini informatif dan tidak bersifat menyesatkan atau provokatif.",
        "Teks ini termasuk konten yang aman dan tidak mengandung disinformasi, fitnah, maupun ujaran kebencian.",
    ],
}

SFT_INSTRUCTION = (
    "Klasifikasikan teks berikut apakah termasuk konten DFK "
    "(Disinformasi, Fitnah, atau Kebencian) dan berikan alasannya."
)

ALPACA_PROMPT_TEMPLATE = """Di bawah ini adalah instruksi yang menjelaskan tugas, dipasangkan dengan input yang memberikan konteks lebih lanjut. Tulis respons yang melengkapi permintaan dengan tepat.

### Instruksi:
{instruction}

### Input:
{input}

### Respons:
{output}"""

# ── Fungsi Preprocessing ──

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"[^\w\s.,!?;:\-'\"\(\)]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def normalize_label(raw_label):
    if pd.isna(raw_label):
        return "Tidak Diketahui"
    raw = str(raw_label).strip().lower()
    if raw in LABEL_MAP:
        return LABEL_MAP[raw]
    for key, value in LABEL_MAP.items():
        if key in raw or raw in key:
            return value
    return "Tidak Diketahui"


def generate_reasoning(label, text):
    if label in REASONING_TEMPLATES:
        return random.choice(REASONING_TEMPLATES[label])
    return f"Teks ini dikategorikan sebagai: {label}."


def validate_alpaca_entry(entry):
    return all(
        key in entry and isinstance(entry[key], str) and len(entry[key].strip()) > 0
        for key in ["instruction", "input", "output"]
    )


def process_hoax_csv(filepath):
    try:
        df = pd.read_csv(filepath, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="latin-1")
    print(f"  Loaded {filepath.name}: {len(df)} rows, columns: {list(df.columns)}")

    text_col = label_col = None
    for col in ["content", "text", "teks", "isi"]:
        if col in df.columns:
            text_col = col; break
    for col in ["classification", "label", "labels", "kategori", "class"]:
        if col in df.columns:
            label_col = col; break
    if text_col is None or label_col is None:
        print(f"  WARNING: Kolom text/label tidak ditemukan di {filepath.name}")
        return []

    title_col = "title" if "title" in df.columns else None
    entries = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {filepath.name}"):
        text = clean_text(str(row[text_col]) if pd.notna(row[text_col]) else "")
        if len(text) < 30:
            continue
        if title_col and pd.notna(row.get(title_col)):
            title = str(row[title_col]).strip()
            if title and title.lower() != "nan":
                text = f"Judul: {title}\n\n{text}"
        label = normalize_label(row[label_col])
        if label == "Tidak Diketahui":
            continue
        reasoning_col = None
        for rc in ["reasoning", "penjelasan", "alasan"]:
            if rc in df.columns:
                reasoning_col = rc; break
        if reasoning_col and pd.notna(row.get(reasoning_col)) and len(str(row[reasoning_col]).strip()) > 30:
            reasoning = str(row[reasoning_col]).strip()
        else:
            reasoning = generate_reasoning(label, text)
        entry = {
            "instruction": SFT_INSTRUCTION,
            "input": text,
            "output": f"Kategori: {label}.\n\nPenjelasan: {reasoning}",
        }
        if validate_alpaca_entry(entry):
            entries.append(entry)
    return entries


def process_hate_speech_csv(filepath):
    try:
        df = pd.read_csv(filepath, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(filepath, encoding="latin-1")
    print(f"  Loaded {filepath.name}: {len(df)} rows")

    text_col = label_col = None
    for col in ["text", "content", "teks"]:
        if col in df.columns:
            text_col = col; break
    for col in ["labels", "label", "class", "classification"]:
        if col in df.columns:
            label_col = col; break
    if text_col is None or label_col is None:
        print(f"  WARNING: Kolom text/label tidak ditemukan di {filepath.name}")
        return []

    entries = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f"  {filepath.name}"):
        text = clean_text(str(row[text_col]) if pd.notna(row[text_col]) else "")
        if len(text) < 20:
            continue
        label = normalize_label(str(row[label_col]))
        if label == "Tidak Diketahui":
            continue
        reasoning = generate_reasoning(label, text)
        entry = {
            "instruction": SFT_INSTRUCTION,
            "input": text,
            "output": f"Kategori: {label}.\n\nPenjelasan: {reasoning}",
        }
        if validate_alpaca_entry(entry):
            entries.append(entry)
    return entries


# ── Pipeline ──

DATA_DIR = PROJECT_ROOT / SFT_DATA_DIR
OUT_DIR = DATA_DIR / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(DATA_DIR.glob("*.csv"))
print(f"CSV files ditemukan: {len(csv_files)}")

all_entries = []
for csv_file in csv_files:
    print(f"\nProcessing: {csv_file.name}")
    try:
        sample_df = pd.read_csv(csv_file, nrows=5)
        columns_lower = [c.lower() for c in sample_df.columns]
    except Exception as e:
        print(f"  SKIP: {e}")
        continue
    if "classification" in columns_lower:
        entries = process_hoax_csv(csv_file)
    elif "labels" in columns_lower or "label" in columns_lower:
        entries = process_hate_speech_csv(csv_file)
    else:
        entries = process_hoax_csv(csv_file)
    print(f"  -> {len(entries)} Alpaca entries")
    all_entries.extend(entries)

print(f"\nTotal entries: {len(all_entries)}")

# ── Apply MAX_ROWS jika TEST_MODE ──
if TEST_MODE and len(all_entries) > MAX_ROWS:
    print(f"\n⚡ TEST_MODE: Membatasi dari {len(all_entries)} → {MAX_ROWS} entries")
    random.seed(42)
    random.shuffle(all_entries)
    all_entries = all_entries[:MAX_ROWS]

# Label distribution
label_counts = {}
for entry in all_entries:
    if "Kategori:" in entry["output"]:
        label = entry["output"].split("Kategori:")[1].split(".")[0].strip()
        label_counts[label] = label_counts.get(label, 0) + 1

print("\nLabel Distribution:")
for label, count in sorted(label_counts.items(), key=lambda x: -x[1]):
    pct = count / len(all_entries) * 100
    print(f"  {label}: {count:,} ({pct:.1f}%)")

# Split
random.seed(42)
shuffled = all_entries.copy()
random.shuffle(shuffled)
split_idx = int(len(shuffled) * (1 - VALIDATION_SPLIT))
train_data, val_data = shuffled[:split_idx], shuffled[split_idx:]
print(f"\nTrain: {len(train_data):,} | Val: {len(val_data):,}")

# Save
train_path = OUT_DIR / "sft_train_alpaca.json"
val_path = OUT_DIR / "sft_val_alpaca.json"
with open(train_path, "w", encoding="utf-8") as f:
    json.dump(train_data, f, indent=2, ensure_ascii=False)
with open(val_path, "w", encoding="utf-8") as f:
    json.dump(val_data, f, indent=2, ensure_ascii=False)

stats = {
    "total_entries": len(all_entries), "train_size": len(train_data),
    "val_size": len(val_data), "label_distribution": label_counts,
    "test_mode": TEST_MODE,
}
with open(OUT_DIR / "sft_stats.json", "w", encoding="utf-8") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"\nData saved to {OUT_DIR}")

## Step 7: SFT Training

Training SFT dengan Unsloth + TRL SFTTrainer.
Jika terputus, jalankan cell ini lagi — auto-resume dari checkpoint terakhir.

In [ ]:
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# ── Load Model ──
print("Loading model...")

model_name = MODEL_NAME
local_path = "/content/hf_cache/" + MODEL_NAME.split("/")[-1]
if os.path.exists(local_path):
    model_name = local_path
    print(f"  Using cached model: {local_path}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
    device_map="auto",
)
print(f"Model loaded: {model_name}")

# ── Auto-load CPT LoRA ──
cpt_lora_path = PROJECT_ROOT / CPT_OUTPUT_DIR / "lora_adapter"
has_cpt_lora = False

if USE_CPT_LORA and cpt_lora_path.exists():
    print(f"\nLoading CPT LoRA from: {cpt_lora_path}")
    try:
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, str(cpt_lora_path), is_trainable=True)
        has_cpt_lora = True
        print("CPT LoRA loaded! SFT will train on top.")
    except Exception as e:
        print(f"WARNING: Failed to load CPT LoRA: {e}")
        print("Training from base model instead.")
else:
    print("No CPT LoRA found, training from base model.")

# EOS token (CRITICAL for SFT)
eos_token = tokenizer.eos_token or ""
print(f"EOS token: {repr(eos_token)}")

# ── Apply LoRA ──
from peft import PeftModel as _PM

if isinstance(model, _PM):
    print("\nCPT LoRA already loaded — SFT trains on top")
else:
    print("\nApplying fresh SFT LoRA...")
    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        bias="none",
        use_rslora=True,
        use_gradient_checkpointing="unsloth",
    )

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total_params:,} ({100*trainable/total_params:.2f}%)")

# ── Load Dataset ──
print("\nLoading SFT dataset...")

train_json = PROJECT_ROOT / SFT_DATA_DIR / "processed" / "sft_train_alpaca.json"
val_json = PROJECT_ROOT / SFT_DATA_DIR / "processed" / "sft_val_alpaca.json"

with open(train_json, "r", encoding="utf-8") as f:
    train_list = json.load(f)
train_dataset = Dataset.from_list(train_list)
print(f"Train: {len(train_dataset):,} samples")

eval_dataset = None
if val_json.exists():
    with open(val_json, "r", encoding="utf-8") as f:
        val_list = json.load(f)
    eval_dataset = Dataset.from_list(val_list)
    print(f"Val:   {len(eval_dataset):,} samples")

# Format prompts
def formatting_prompts_func(examples):
    texts = []
    for instr, inp, out in zip(examples["instruction"], examples["input"], examples["output"]):
        text = ALPACA_PROMPT_TEMPLATE.format(instruction=instr, input=inp, output=out) + eos_token
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
if eval_dataset:
    eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)

print(f"\nSample prompt (truncated):\n{train_dataset['text'][0][:300]}...")

# ── W&B Init ──
wandb.init(
    project=WANDB_PROJECT,
    name=WANDB_RUN_FULL,
    config={
        "model": MODEL_NAME, "epochs": NUM_EPOCHS,
        "batch_size": BATCH_SIZE * GRADIENT_ACCUMULATION,
        "lr": LEARNING_RATE, "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
        "cpt_lora": has_cpt_lora,
        "test_mode": TEST_MODE, "max_rows": MAX_ROWS if TEST_MODE else "all",
    },
    tags=["sft", "qwen3", "dfk", "alpaca"] + (["test"] if TEST_MODE else []),
    reinit=True,
)

# ── Training ──
output_dir = PROJECT_ROOT / SFT_OUTPUT_DIR

training_args = TrainingArguments(
    output_dir=str(output_dir),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=True,
    bf16=False,
    seed=42,
    eval_strategy="steps" if eval_dataset else "no",
    eval_steps=EVAL_STEPS if eval_dataset else None,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=3,
    load_best_model_at_end=bool(eval_dataset),
    metric_for_best_model="eval_loss" if eval_dataset else None,
    logging_steps=10,
    report_to="wandb",
    optim="adamw_8bit",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

# Auto-resume dari checkpoint
last_ckpt = None
if output_dir.exists():
    ckpts = sorted(output_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[-1]))
    if ckpts:
        last_ckpt = str(ckpts[-1])
        print(f"Resuming from checkpoint: {last_ckpt}")

if TEST_MODE:
    print(f"\n⚡ TEST_MODE: {NUM_EPOCHS} epoch, {len(train_dataset)} train samples")
print("\nStarting SFT training...")
trainer.train(resume_from_checkpoint=last_ckpt)
print("\nTraining complete!")

# ── Save LoRA Adapter ──
lora_save = str(output_dir / "lora_adapter")
os.makedirs(lora_save, exist_ok=True)
model.save_pretrained(lora_save)
tokenizer.save_pretrained(lora_save)
print(f"LoRA adapter saved: {lora_save}")

wandb.finish()

## Step 8: Inference Test

In [ ]:
if RUN_INFERENCE_TEST:
    print("=" * 60)
    print("INFERENCE TEST")
    print("=" * 60)

    FastLanguageModel.for_inference(model)

    test_samples = [
        {"input": "Vaksin COVID-19 mengandung microchip untuk melacak pergerakan manusia. Bill Gates mendalangi ini semua.", "expected": "Hoax"},
        {"input": "Pemerintah Kominfo mengklarifikasi bahwa berita tentang vaksin adalah tidak benar.", "expected": "Bukan DFK"},
        {"input": "Orang-orang dari suku tertentu adalah penyebab semua masalah di negara ini. Mereka harus diusir!", "expected": "Ujaran Kebencian"},
    ]

    instruction = SFT_INSTRUCTION

    for i, sample in enumerate(test_samples):
        prompt = ALPACA_PROMPT_TEMPLATE.format(
            instruction=instruction, input=sample["input"], output=""
        )
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048).to(model.device)
        outputs = model.generate(
            **inputs, max_new_tokens=256, temperature=0.7, top_p=0.9,
            do_sample=True, eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        response = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "### Respons:" in response:
            response = response.split("### Respons:")[-1].strip()

        predicted = "Unknown"
        for lab in ["Hoax", "Disinformasi", "Misinformasi", "Ujaran Kebencian", "Fitnah", "Bukan DFK"]:
            if f"Kategori: {lab}" in response:
                predicted = lab; break

        status = "CORRECT" if predicted == sample["expected"] else "WRONG"
        print(f"\nTest {i+1}:")
        print(f"  Input: {sample['input'][:80]}...")
        print(f"  Expected: {sample['expected']}")
        print(f"  Predicted: {predicted} [{status}]")
        print(f"  Output: {response[:150]}...")
else:
    print("Inference test disabled. Set RUN_INFERENCE_TEST = True.")

## Step 9: Merge LoRA → Model Final (16-bit)

Merge LoRA adapter ke base model menjadi **model penuh 16-bit**.
Ini adalah **output akhir** yang siap digunakan langsung — tanpa perlu base model + adapter terpisah.
Model disimpan di Google Drive (`outputs/sft/merged_model/`).

In [ ]:
if SAVE_MERGED_16BIT:
    print("=" * 60)
    print("MERGE LoRA → MODEL FINAL (16-bit)")
    print("=" * 60)

    merged_path = str(PROJECT_ROOT / SFT_OUTPUT_DIR / "merged_model")
    print(f"\nMerging LoRA into base model...")
    print("(Proses ini memerlukan beberapa menit — dequantize layer by layer)")
    model.save_pretrained_merged(
        merged_path,
        tokenizer,
        save_method="merged_16bit",
    )
    total_gb = sum(f.stat().st_size for f in Path(merged_path).rglob("*") if f.is_file()) / 1e9
    print(f"\n✅ Model final tersimpan: {merged_path}")
    print(f"   Size: {total_gb:.2f} GB")
    print(f"   Files:")
    for f in sorted(Path(merged_path).glob("*")):
        if f.is_file():
            print(f"     {f.name} ({f.stat().st_size / 1e6:.1f} MB)")

    print("\n📌 Model ini siap dipakai langsung:")
    print(f"   from transformers import AutoModelForCausalLM, AutoTokenizer")
    print(f'   model = AutoModelForCausalLM.from_pretrained("{merged_path}")')
    print(f'   tokenizer = AutoTokenizer.from_pretrained("{merged_path}")')
else:
    print("SAVE_MERGED_16BIT = False.")
    print("Hanya LoRA adapter yang tersimpan.")
    print("Set SAVE_MERGED_16BIT = True di Step 1 untuk menyimpan model final.")

## Step 10: Verify Output

In [ ]:
print("=" * 60)
print("SFT PIPELINE COMPLETE!")
print("=" * 60)

# Check LoRA adapter
lora_path = PROJECT_ROOT / SFT_OUTPUT_DIR / "lora_adapter"
if lora_path.exists():
    total_size = sum(f.stat().st_size for f in lora_path.rglob("*") if f.is_file()) / 1024 / 1024
    print(f"\n📦 LoRA Adapter: {lora_path} ({total_size:.1f} MB)")
else:
    print("\n❌ LoRA adapter tidak ditemukan")

# Check merged model
merged_path = PROJECT_ROOT / SFT_OUTPUT_DIR / "merged_model"
if merged_path.exists():
    total_gb = sum(f.stat().st_size for f in merged_path.rglob("*") if f.is_file()) / 1e9
    print(f"\n📦 Model Final (16-bit): {merged_path} ({total_gb:.2f} GB)")
    for f in sorted(merged_path.glob("*.safetensors")):
        print(f"     {f.name} ({f.stat().st_size / 1e9:.2f} GB)")
else:
    print("\n⚠️  Model final tidak ada (SAVE_MERGED_16BIT = False?)")

print(f"\nModel: {MODEL_NAME}")
print(f"CPT LoRA: {'Loaded' if has_cpt_lora else 'Not used'}")

if TEST_MODE:
    print("\n⚡ Ini hasil TEST MODE — untuk full training, set TEST_MODE = False")

---
## Summary

### Output
```
outputs/sft/
├── lora_adapter/              ← LoRA adapter (untuk incremental training)
│   ├── adapter_config.json
│   ├── adapter_model.safetensors
│   └── tokenizer files
└── merged_model/              ← MODEL FINAL (16-bit, siap pakai)
    ├── config.json
    ├── model-00001-of-XXXXX.safetensors
    ├── ...
    ├── tokenizer.json
    └── tokenizer_config.json
```

### Pipeline
```
Base Model (Qwen3-8B 4-bit)
    │
    ├── CPT LoRA (domain adaptation)
    │
    ├── SFT LoRA (task training)
    │
    └── Merge LoRA → Model Final 16-bit  ← OUTPUT AKHIR
        (tersimpan di Google Drive)
```

### Test Mode vs Full Training
| Setting | Test Mode | Full Training |
|---------|-----------|---------------|
| `TEST_MODE` | `True` | `False` |
| `MAX_ROWS` | 1000 | semua data |
| `NUM_EPOCHS` | 1 | 5 |
| Waktu | ~5 menit | 2-4 jam |

### Cara Pakai Model Final

```python
from transformers import AutoModelForCausalLM, AutoTokenizer

# Load dari Google Drive
model = AutoModelForCausalLM.from_pretrained("/path/to/outputs/sft/merged_model")
tokenizer = AutoTokenizer.from_pretrained("/path/to/outputs/sft/merged_model")

# Inference
prompt = "Klasifikasikan teks berikut: ..."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
output = model.generate(**inputs, max_new_tokens=256)
print(tokenizer.decode(output[0], skip_special_tokens=True))
```